<div style="display: flex; align-items: center; gap: 18px; margin-bottom: 15px;">
  <img src="https://files.codebasics.io/v3/images/sticky-logo.svg" alt="Codebasics Logo" style="display: inline-block;" width="130">
  <h1 style="font-size: 34px; color: #1f4e79; margin: 0; display: inline-block;">Codebasics Practice Room - Data Engineering Bootcamp </h1>
</div>

## 🧑🏼‍🔧 Setup

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
from delta.tables import DeltaTable

In [0]:
# --------------------------------------------
# ⚙️ Databricks Unity Catalog Setup (Auto)
# --------------------------------------------
from pyspark.sql import SparkSession

catalog_name = "practice_db_catalog"
schema_name = "airbnb"
volume_name = "data_volume"

# 1️⃣ Create Catalog if not exists
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
print(f"✅ Catalog `{catalog_name}` ready.")

# 2️⃣ Create Schema (Database) if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"✅ Schema `{schema_name}` created inside `{catalog_name}`.")

# 3️⃣ Create Volume if not exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
print(f"✅ Volume `{volume_name}` created inside `{catalog_name}.{schema_name}`")

# 4️⃣ Set current context
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {schema_name}")

# 5️⃣ Define volume-backed paths
base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/airbnb"
raw_path = f"{base_path}/raw"
clean_path = f"{base_path}/clean"
silver_path = f"{base_path}/silver"

# 6️⃣ Create directories inside the volume
dbutils.fs.mkdirs(raw_path)
dbutils.fs.mkdirs(clean_path)
dbutils.fs.mkdirs(silver_path)

print("✅ Paths initialized successfully:")
print(f"Raw: {raw_path}")
print(f"Clean: {clean_path}")
print(f"Silver: {silver_path}")


✅ Catalog `practice_db_catalog` ready.
✅ Schema `airbnb` created inside `practice_db_catalog`.
✅ Volume `data_volume` created inside `practice_db_catalog.airbnb`
✅ Paths initialized successfully:
Raw: /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw
Clean: /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/clean
Silver: /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver


In [0]:
# 🧮 Generate Airbnb listings dataset (Spark-native version)
from pyspark.sql import Row
import random, datetime

# --------------------------------
# Configuration
# --------------------------------

random.seed(42) # ✅ reproducibility

num_records = 600  # Adjust as needed

amenities_pool = [
    "Wifi", "Kitchen", "Washer", "Dryer", "TV", "Essentials", "Air conditioning",
    "Heating", "Pool", "Hot tub", "Balcony", "Garden", "Parking", "Fireplace",
    "Sea view", "Mountain view", "Pet friendly", "Gym", "Breakfast", "Workspace"
]

property_types = [
    "Studio Apartment", "Private Room", "Entire Home", "Cottage", "Villa",
    "Cabin", "Loft", "Guest Suite", "Bungalow", "Condo"
]

cities = ["Mumbai", "Bangalore", "Hyderabad", "Chennai", "Pune", "Delhi", "Goa"]
boolean_variants = [True, False, "true", "false", "Yes", "No", "yes", "no", "TRUE", "FALSE"]

# --------------------------------
# Generate data as list of Rows
# --------------------------------
data = []
for i in range(1, num_records + 1):
    created_date = datetime.date(2025, 1, 1) + datetime.timedelta(days=random.randint(0, 300))
    last_booked_date = created_date + datetime.timedelta(days=random.randint(1, 60))

    data.append(Row(
        id=100 + i,
        name=f"{random.choice(['Cozy', 'Modern', 'Luxury', 'Spacious', 'Budget'])} "
             f"{random.choice(property_types)} in {random.choice(cities)}",
        city=random.choice(cities),
        price_per_night=random.randint(1000, 10000),
        amenities=random.sample(amenities_pool, random.randint(3, 8)),
        has_parking=random.choice(boolean_variants),
        is_superhost=random.choice(boolean_variants),
        created_date=str(created_date),
        last_booked_date=str(last_booked_date)
    ))

# --------------------------------
# Convert to Spark DataFrame
# --------------------------------
df_raw = spark.createDataFrame(data)

# --------------------------------
# Write directly to UC Volume (JSON format)
# --------------------------------
raw_path = "/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json"

df_raw.write.mode("overwrite").json(raw_path)
print(f"✅ Successfully generated {num_records} Airbnb listings and saved to:")
print(f"📂 {raw_path}")

# --------------------------------
# Quick sanity check
# --------------------------------


✅ Successfully generated 600 Airbnb listings and saved to:
📂 /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json



# ❓ Scenario Question: Airbnb — Clean Listing Amenities (PySpark) [Easy]



## 🗂️ Scenario

You are working with raw **Airbnb listing data** ingested from multiple sources.  
Each listing contains property details and a **nested list of amenities**.  
The goal is to **clean, normalize, and store** this data for downstream analysis.

The data is available as a JSON file (`listings.json`) in the **Bronze layer**, which now needs to be transformed into a clean **Silver Delta Table**.

---

## 🎯 Task

Perform the following transformations:

1. **Read** the input data from `listings.json` using Spark.  
2. **Explode** the `amenities` array so that each row contains a single amenity.  
3. **Normalize** boolean-like columns (e.g., `"true"`, `"false"`, `"yes"`, `"no"`) into proper boolean (`True` / `False`) Spark data types.  
4. **Rename** or select only the relevant columns for downstream use.  
5. **Save** the cleaned DataFrame in **Delta format** to the **Silver layer** path:  
   `/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver/listings.json` 

---

## 🧩 Assumptions

- The input file `listings.json` exists in the **Bronze** path:  
  `/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json`
- The `amenities` field may contain an array or a stringified array.  
- Boolean columns may contain values like `"TRUE"`, `"Yes"`, `"0"`, `"1"`, etc.  
- The final cleaned DataFrame should contain only essential columns:  
  `id`, `name`, `amenity`, `has_parking`, and `is_superhost`.  
- Handle missing or malformed columns gracefully (e.g., cast to `null`).  

---

## 📦 Deliverables

- **Output Format:** Delta table written to Silver  
- **Output Path:** `/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver/listings.json`

| **Expected Columns** | `id`, `name`, `amenity`, `has_parking`, `is_superhost` |

---

## 🧠 Notes

- Use `pyspark.sql.functions.explode()` to expand the amenities array.  
- Use `F.when()` or `F.col().cast("boolean")` for boolean normalization.  
- Use clear column aliases for readability.  
- Validate the write by reading from the Silver path and displaying the first few rows.

---

## 🧩 Example Output (simplified)

| id  | name               | amenity          | has_parking | is_superhost |
|-----|--------------------|------------------|--------------|---------------|
| 101 | Cozy Beach House   | Wifi             | true         | false         |
| 101 | Cozy Beach House   | Ocean View       | true         | false         |
| 102 | City Apartment     | Air Conditioning | false        | true          |


## 🛢️Input data

In [0]:
display(df_raw.limit(5))

id,name,city,price_per_night,amenities,has_parking,is_superhost,created_date,last_booked_date
101,Luxury Cottage in Bangalore,Bangalore,2679,"List(Gym, Washer, Fireplace, Kitchen, Wifi, Pet friendly, Dryer, Workspace)",TRUE,FALSE,2025-02-27,2025-03-01
102,Modern Bungalow in Chennai,Bangalore,8359,"List(Pool, Wifi, Essentials, Fireplace, Balcony, TV, Washer)",false,No,2025-01-14,2025-02-19
103,Spacious Private Room in Hyderabad,Goa,6635,"List(Pool, Kitchen, Sea view, Dryer, Parking, Breakfast, Workspace)",Yes,FALSE,2025-02-22,2025-02-28
104,Modern Private Room in Mumbai,Delhi,4733,"List(Washer, Heating, Dryer, Parking, Pool)",no,No,2025-07-05,2025-08-11
105,Luxury Cottage in Delhi,Hyderabad,2169,"List(Essentials, Gym, Heating, Workspace, Sea view, Air conditioning, TV)",TRUE,false,2025-03-25,2025-04-18


# 📝 Your Solution

In [0]:
# ✍️ Your Solution Here
from pyspark.sql import functions as F
from pyspark.sql.functions import explode

# Steps:
# 1. Read the JSON file
# 2. Explode the amenities
# 3. Normalize boolean-like fields and retrun the dataframe


In [0]:
# read the data from bronze volume 

df = spark.read.format("json") \
    .load("/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json")

In [0]:
df_exploded = df.withColumn("amenity", explode("amenities"))
display(df_exploded.limit(5))

amenities,city,created_date,has_parking,id,is_superhost,last_booked_date,name,price_per_night,amenity
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Sea view
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Hot tub
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Pool
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Garden
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Heating


In [0]:
df_exploded.select("has_parking").distinct().show()

+-----------+
|has_parking|
+-----------+
|      false|
|       TRUE|
|        Yes|
|        yes|
|       true|
|      FALSE|
|         No|
|         no|
+-----------+



In [0]:
parking = F.lower(F.trim(F.col("has_parking").cast("string")))

df_clean = df_exploded.withColumn(
    "has_parking",
    F.when(parking.isin("true","True", "Yes", "yes", "1"), True)
    .when(parking.isin("false","False","No", "no", "0"), False)
    .otherwise(None)
    .cast("boolean")
)

display(df_clean.limit(5))

amenities,city,created_date,has_parking,id,is_superhost,last_booked_date,name,price_per_night,amenity
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Sea view
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Hot tub
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Pool
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Garden
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,false,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Heating


In [0]:
superhost = F.lower(F.trim(F.col("is_superhost").cast("string")))
df_clean = df_clean.withColumn(
    "is_superhost",
    F.when(superhost.isin("True","true","Yes","yes","1"), True)
    .when(superhost.isin("False","false","No","no","0"), False)
    .otherwise(None)
    .cast("Boolean")
)

display(df_clean.limit(5))

amenities,city,created_date,has_parking,id,is_superhost,last_booked_date,name,price_per_night,amenity
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Sea view
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Hot tub
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Pool
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Garden
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,false,477,true,2025-10-12,Modern Studio Apartment in Pune,6965,Heating


In [0]:
airbnb_df = df_clean.select(
    F.col("id"),
    F.col("name"),
    F.col("amenity"),
    F.col("has_parking"),
    F.col("is_superhost")
)

In [0]:
display(airbnb_df.limit(5))

id,name,amenity,has_parking,is_superhost
476,Budget Bungalow in Bangalore,Sea view,false,true
476,Budget Bungalow in Bangalore,Hot tub,false,true
476,Budget Bungalow in Bangalore,Pool,false,true
476,Budget Bungalow in Bangalore,Garden,false,true
477,Modern Studio Apartment in Pune,Heating,false,true


In [0]:
airbnb_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save("/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver/slv_listings")

## 🔍 Validation Questions

After creating the final DataFrame (`df_final`), answer these to check your understanding:

1. How many amenities are listed for the property with **ID = 101**?  
2. How many listings have **`is_superhost = true`**?  
3. What are the **unique amenities** available for listing **ID = 103**?  
4. Count how many listings have **`has_parking = true`**.  
5. For each listing, how many total amenities are available? (Hint: use `groupBy().count()`.)

In [0]:
# How many amenities are listed for the property with ID = 101?

cnt_amenities = airbnb_df.filter(F.col("id") == 101).count()
print(f"Number of amenities for property with ID = 101: {cnt_amenities}")



Number of amenities for property with ID = 101: 8


In [0]:
# How many listings have is_superhost = true?
cnt_list_superhost = airbnb_df.filter(F.col("is_superhost") == "true").select("id").distinct().count()
print(f"Number of listings have is_superhost True: {cnt_list_superhost}")


Number of listings have is_superhost True: 316


In [0]:
# What are the unique amenities available for listing ID = 103?

unique_amenities = airbnb_df.filter(F.col("id") == 103).select("amenity").distinct().count()
print(f"Unique amenities for listing ID = 103: {unique_amenities}")

Unique amenities for listing ID = 103: 7


In [0]:
# Count how many listings have has_parking = true.

cnt_parking = airbnb_df.filter(F.col("has_parking") == "true").select("id").distinct().count()
print(f"Number of listings have has_parking True: {cnt_parking}")

Number of listings have has_parking True: 290


In [0]:
# For each listing, how many total amenities are available? 

amenity_counts = airbnb_df.groupBy("id").agg(F.count("amenity").alias("amenity_count"))
display(amenity_counts)

id,amenity_count
480,3
501,4
517,4
519,7
490,6
498,7
510,6
514,7
530,4
542,4
